# 🧪 Test Colab — est-ce que j'ai bien un GPU NVIDIA ?

Petit notebook pour **vérifier** que Colab marche, avant de lancer un vrai projet.
5 minutes, 5 cellules. `Run All` et tu sais.

## 🔒 Il ne touche pas à ton Drive

Tout est sur **ton disque**. Sur Colab, tu envoies le CSV **à la main**
(`files.upload()`) : Google ne voit que ce fichier, et **rien ne reste** après la
session.

`drive.mount()` n'est **pas** utilisé — il monterait ton « Mon Drive » **en entier**
(photos, vidéos, documents), sans aucun moyen de le limiter à un dossier.

> 💡 **Réflexe** : avant de lancer un notebook que tu n'as pas écrit, cherche
> `drive.mount` dedans.

## Comment se connecter au T4

Extension **`Google Colab`** requise (`Ctrl+Shift+X`, éditeur **Google**).

**`Select Kernel`** (en haut à droite) → `Select Another Kernel...` → `Colab` →
connexion Google → `Auto Connect`.

Si tu récupères un CPU, refais-le en prenant **`New Colab Server` → `GPU` → `T4`**.

> ⚠️ Le menu `Runtime → Change runtime type` n'existe **que sur le Colab web**.
> Dans VS Code, le GPU se choisit **à la création du serveur**.

## Tes 3 noyaux

| Choix | Matériel |
|---|---|
| `Python 3` | ton CPU |
| `Python (GPU ROCm)` | ton Radeon 880M |
| `Colab` | NVIDIA T4 distant |

## 1. Où je tourne, et sur quoi ?

In [ ]:
import torch

try:
    import google.colab
    DANS_COLAB = True
except ImportError:
    DANS_COLAB = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("J'execute sur :", "Google Colab (a distance)" if DANS_COLAB else "ton PC")
print(f"torch         : {torch.__version__}")
print(f"device        : {device}")

if device.type == "cuda":
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU           : {torch.cuda.get_device_name(0)}  ({vram:.1f} Go de VRAM)")
    print("\n>>> GPU OK.")
elif DANS_COLAB:
    print("\n>>> Colab t'a donne un CPU, pas un GPU.")
    print("    Select Kernel -> Select Another Kernel... -> Colab")
    print("    -> New Colab Server -> GPU -> T4")
else:
    print("\n>>> Noyau CPU. Pour ton Radeon : Select Kernel -> Python (GPU ROCm)")

## 2. Les données

**Une seule source de vérité : ton dépôt GitHub.** La même URL sur ton PC et sur Colab —
donc **le même fichier, le même résultat, toujours**. Aucun clic, `Run All` d'un bout à
l'autre.

```
donnees/Advertising.csv   ->   tu le pousses   ->   le notebook le relit par son URL
```

Le dossier `donnees/` est **ta source** : c'est là que tu déposes tes fichiers. Git les
transporte, l'URL les relit.

### ⚠️ La seule règle

> **Tu modifies un CSV dans `donnees/` → tu pousses avant de relancer.**

Sinon le notebook lit l'ancienne version, **sans rien dire**. C'est le prix d'avoir une
source unique — et c'est aussi ce qui garantit que ton PC et Colab ne divergeront jamais.

### 🔒 Pourquoi c'est le choix sûr

Colab fait un simple téléchargement, comme un navigateur. **Ton Drive n'est jamais touché**,
aucune autorisation n'est demandée, rien n'est monté.

### ❌ Ce qui ne marche PAS, et pourquoi

**`files.upload()`** → *« Upload widget is only available when the cell has been executed in
the current browser session »*. Elle ouvre une fenêtre **JavaScript** qui a besoin de
l'interface **web** de Colab. Dans VS Code elle n'existe pas : le widget s'affiche mais
reste mort, et la cellule attend indéfiniment. **Aucune astuce ne contourne ça** — c'est
dans les *Known Issues* de l'extension.

**`drive.mount()`** → il monterait ton « Mon Drive » **en entier** (photos, vidéos,
documents), sans aucun moyen de le limiter à un dossier.

**Lire un chemin local depuis Colab** → impossible aussi : *« the remote Colab runtime
cannot access local files »*. La machine Google est **vide**, ton dossier ne part pas avec
le notebook. Seul le **code** est envoyé.

> 💡 **Le réflexe à garder** : avant de lancer un notebook que tu n'as pas écrit, cherche
> `drive.mount` dedans.
>
> Ce que tu as déjà autorisé :
> [myaccount.google.com/permissions](https://myaccount.google.com/permissions)

In [ ]:
import pandas as pd

# Une seule source de verite : ton depot GitHub public.
# Meme URL sur ton PC et sur Colab -> meme fichier, meme resultat, toujours.
GITHUB = ("https://raw.githubusercontent.com/RobinsanKiritheepan/"
          "Cours_Python/main/Cours_IA/Deep_learning/Google_Colab/donnees")


def source(nom):
    """L'URL du fichier sur ton depot.

    Colab telecharge 3,8 Ko par HTTP et rien d'autre : ton Drive n'est jamais
    touche, aucune autorisation demandee.

    files.upload() ne marche pas dans VS Code (il lui faut l'interface web de Colab).
    drive.mount() exposerait TOUT ton Drive. Ni l'un ni l'autre ici.
    """
    return f"{GITHUB}/{nom}"


chemin = source("Advertising.csv")
print("Lecture depuis GitHub :")
print(chemin)

df = pd.read_csv(chemin)
print(f"\n{len(df)} lignes, {len(df.columns)} colonnes")
df.head()

## 3. Le GPU calcule-t-il vraiment ?

Détecter un GPU et **s'en servir** sont deux choses différentes. On entraîne un petit
réseau et on regarde si la perte descend.

In [ ]:
import time

from torch import nn

torch.manual_seed(42)

# les donnees partent sur le device (GPU si dispo)
X = torch.tensor(df[["TV", "radio", "newspaper"]].values, dtype=torch.float32)
y = torch.tensor(df[["sales"]].values, dtype=torch.float32)
X = ((X - X.mean(0)) / X.std(0)).to(device)   # normalisation, sinon ca n'apprend pas
y = y.to(device)

modele = nn.Sequential(
    nn.Linear(3, 32), nn.ReLU(),
    nn.Linear(32, 1),
).to(device)                                  # <- le modele part sur le GPU

perte_fn = nn.MSELoss()
opt = torch.optim.Adam(modele.parameters(), lr=0.05)

t0 = time.perf_counter()
for epoch in range(500):
    opt.zero_grad()
    perte = perte_fn(modele(X), y)
    perte.backward()
    opt.step()
    if epoch == 0:
        perte_debut = perte.item()
if device.type == "cuda":
    torch.cuda.synchronize()
duree = time.perf_counter() - t0

print(f"perte : {perte_debut:.2f}  ->  {perte.item():.2f}")
print(f"500 epochs en {duree:.2f} s sur {device}")
print(f"le modele est sur : {next(modele.parameters()).device}")
print("\n>>> " + ("OK, ca apprend." if perte.item() < perte_debut else "ECHEC : la perte ne descend pas."))

## 4. Le verdict

In [ ]:
ou = "Google Colab" if DANS_COLAB else "ton PC"
materiel = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"

print("=" * 50)
print(f"  Endroit  : {ou}")
print(f"  Materiel : {materiel}")
print(f"  Vitesse  : {duree:.2f} s pour 500 epochs")
print(f"  Drive    : jamais monte")
print("=" * 50)
print("\nNote la vitesse, change de noyau, relance tout, compare.")

## 5. À quoi t'attendre

**Ce réseau fait 161 paramètres. Le GPU est PLUS LENT que ton CPU.**
C'est normal, ce n'est pas un bug : le temps d'envoyer les données vers la carte dépasse
le temps de calcul.

**Mesuré avec CE notebook, sur ta machine, le 16/07/2026** (500 epochs) :

| Noyau | Temps |
|---|---|
| `Python 3` (CPU) | **0,27 s** |
| `Python (GPU ROCm)` (Radeon 880M) | **1,09 s** ← **4× plus lent** |

La perte est identique des deux côtés (211,60 → 0,20) : ton code est reproductible, seule
la vitesse change.

Un GPU, c'est un poids lourd : inutile pour livrer une lettre. Il gagne sur les **CNN sur
images**, les **transformers**, et les **batchs ≥ 32**.

| Ce que tu fais | Utilise |
|---|---|
| ce test, CSV, sklearn, petits réseaux | **ton CPU** (`Python 3`) |
| CNN sur images, transformers | **Colab (T4)** |
| tester ton Radeon | `Python (GPU ROCm)` |

**Ce notebook ne sert pas à aller vite** — il sert à vérifier que **la plomberie marche**
avant que tu en aies vraiment besoin. Le jour du premier CNN, tout sera déjà en place.

---

### ⚠️ Pièges connus

**1.** Le noyau `Python (GPU ROCm)` **se fige au démarrage de temps en temps** (0 % de CPU,
rien ne s'affiche). Stack ROCm sur Windows, en preview. `Restart Kernel` et ça repart.

**2.** L'auth de l'extension Colab a des bugs ouverts (`Exchange timeout exceeded`). Si
ça coince : réessaie, ou passe par `Open Colab Web` (il faudra uploader ce notebook à la
main).

**3.** Sur Colab, `/content` est **vidé à chaque fermeture de session**. Tu devras
renvoyer le CSV. C'est le défaut **et** la garantie que rien ne traîne chez Google.